# 02 - Bronze: network KPI measurements (12 batches)
Auto Loader (cloudFiles) with schemaLocation and checkpointLocation. Appends to Bronze and never edits values.
All columns stay as **text**, so blanks, sentinel numbers and odd values are kept exactly as delivered.
Each row also gets `source_file`, `source_file_modified`, `ingest_ts` and a `row_hash` of all source columns (for lineage and duplicate checks).

## 1. Settings

In [ ]:
from pyspark.sql import functions as F

vol   = "/Volumes/capstone6_telecom/telecom/ext_raw_landing"
src   = f"{vol}/adls/telemetry_data/network_kpi_measurements/"
table = "capstone6_telecom.bronze.bronze_network_kpi_measurements"

## 2. Load into Bronze

In [ ]:
df = (spark.readStream.format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .option("header", "true")
      .option("cloudFiles.schemaLocation", f"{vol}/_schemas/kpi_v2")
      .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
      .load(src))

src_cols = [c for c in df.columns if c != "_rescued_data"]

df = (df
      .withColumn("source_file", F.col("_metadata.file_path"))
      .withColumn("source_file_modified", F.col("_metadata.file_modification_time"))
      .withColumn("ingest_ts", F.current_timestamp())
      .withColumn("row_hash", F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c), F.lit("")) for c in src_cols]), 256)))

(df.writeStream
   .option("checkpointLocation", f"{vol}/_checkpoints/kpi_v2")
   .option("mergeSchema", "true")
   .trigger(availableNow=True)
   .toTable(table)
   .awaitTermination())

## 3. Check the result (the source files have 11,894 rows in total)

In [ ]:
display(spark.sql(f"SELECT COUNT(*) AS total_rows FROM {table}"))

In [ ]:
display(spark.sql(f"SELECT source_file, COUNT(*) AS rows FROM {table} GROUP BY source_file ORDER BY source_file"))

Running this notebook again adds no rows, because the checkpoint remembers the files already loaded.

To start over: drop the table and delete the `_checkpoints/kpi_v2` and `_schemas/kpi_v2` folders from the volume in the Catalog UI.